## Historical Aerial Photograph Index Preparation

This notebook cleans and standardizes the original Minnesota Historical Aerial Photographs Online (MHAPO) point feature class and produces two datasets:
- A cleaned feature class containing the original useful metadata, corrected attributes, and unique photograph identifiers.
- A blank georeferencing index containing the subset of attributes required for the batch georeferencing workflow.

Both datasets share a common unique identifier (source_id) to support joins and subsequent updates.

The original source data are preserved without modification. Processing results and georeferencing status are managed separately from these datasets.

### Planning
- Preserve the original row number for provenance immediately after loading.
- Perform all cleaning and deduplication.
- Sort the final cleaned table.
- Assign permanent source_id values.
- Generate and validate cty_year_id.
- Derive both output tables from that same cleaned GeoDataFrame.

#### Sequence
- 01 Setup and configuration
- 02 Load and inspect source data
- 03 Remove empty fields and standardize field names
- 04 Normalize attribute values
- 05 Spatial county assignment and validation
- 06 Correct known URL errors
- 07 Investigate and resolve duplicate records
- 08 Sort records and generate permanent IDs
- 09 Calculate derived attributes
- 10 Construct the blank georeferencing index
- 11 Validate both output datasets
- 12 Export cleaned feature class and blank index

In [2]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

# Locate the repository root.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\eu01251115\LocalProjects\batch_georef


In [3]:
# Configure file paths
# This is separate from config.yml, to isolate this from batch processing and facilitate manual re-runs

DATA_DIR = PROJECT_ROOT / "data" / "historic_airphoto_index"

SOURCE_PATH = (DATA_DIR / "shapefiles" / "umn_airphoto_index_original.shp")

COUNTIES_PATH = (DATA_DIR / "shapefiles" / "Minnesota_Counties.shp")

CLEANED_PATH = DATA_DIR / "umn_index_cleaned.gpkg"

INDEX_PATH = DATA_DIR / "georef_index_blank.gpkg"

LAYER = "photos"

for path in [SOURCE_PATH, COUNTIES_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

In [4]:
# Load and inspect source data

gdf_original = gpd.read_file(SOURCE_PATH)
counties = gpd.read_file(COUNTIES_PATH)

print("Photo records:", len(gdf_original))
print("Photo index CRS:", gdf_original.crs)

print("County records:", len(counties))
print("County CRS:", counties.crs)

Photo records: 115749
Photo index CRS: EPSG:3857
County records: 87
County CRS: EPSG:3857


In [5]:
# Inspect geometry types and quality.
for name, dataset in [
    ("Photos", gdf_original),
    ("Counties", counties),
]:
    print(f"\n{name}")
    print(dataset.geom_type.value_counts())
    print("Missing geometries:", dataset.geometry.isna().sum())
    print("Empty geometries:", dataset.geometry.is_empty.sum())
    print("Invalid geometries:", (~dataset.geometry.is_valid).sum())


Photos
Point    115749
Name: count, dtype: int64
Missing geometries: 0
Empty geometries: 0
Invalid geometries: 0

Counties
Polygon         85
MultiPolygon     2
Name: count, dtype: int64
Missing geometries: 0
Empty geometries: 0
Invalid geometries: 0


In [16]:
# Create a working copy
gdf = gdf_original.copy().reset_index(drop=True)

# Preserve the original position of each record.
gdf["source_row"] = gdf.index

In [17]:
# Standardize whitespace-only strings as missing values.
text_columns = gdf.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    gdf[col] = gdf[col].replace(
        r"^\s*$", pd.NA, regex=True
    )

In [18]:
# Inspect the schema - profile of the table

field_summary = pd.DataFrame({
    "dtype": gdf.dtypes.astype(str),
    "null_count": gdf.isna().sum(),
    "null_pct": (gdf.isna().mean() * 100).round(2),
    "unique_values": gdf.nunique(dropna=True),
})

display(field_summary)

,dtype,null_count,null_pct,unique_values
flight_cod,str,0,0.00,132
url_link,str,0,0.00,115563
photo_id,str,0,0.00,106857
lat,float64,0,0.00,55847
long_,float64,0,0.00,69028
year_,float64,0,0.00,49
objectid_1,int64,0,0.00,111070
in_fid,int64,0,0.00,111070
near_fid,int64,0,0.00,88
near_dist,float64,0,0.00,1712


In [19]:
# locate completely empty columns

empty_fields = [
    col
    for col in gdf.columns
    if col != gdf.geometry.name
    and gdf[col].isna().all()
]

print("Entirely empty fields:")
print(empty_fields)

Entirely empty fields:
['metdivfp10', 'decade_dat', 'created_us', 'created_da']


In [20]:
# drop completely empty columns

gdf = gdf.drop(columns=empty_fields)

print("Fields removed:", len(empty_fields))
print("Remaining fields:", len(gdf.columns))

Fields removed: 4
Remaining fields: 34


In [21]:
FIELD_RENAME = {
    "url_link": "url",
    "year_": "year",
    "name10": "county",
    "flight_cod": "flight_code",
    "long_": "longitude_og",
    "lat": "latitude_og"
}

# Confirm expected fields exist.
missing_fields = set(FIELD_RENAME) - set(gdf.columns)

if missing_fields:
    raise KeyError(
        f"Expected fields are missing: {missing_fields}"
    )

In [22]:
gdf = gdf.rename(columns=FIELD_RENAME)

if not gdf.columns.is_unique:
    raise ValueError("Renaming created duplicate field names.")

display(gdf.head())

,flight_code,url,photo_id,latitude_og,longitude_og,year,objectid_1,in_fid,near_fid,near_dist,objectid_2,statefp10,countyfp10,countyns10,geoid10,county,namelsad10,lsad10,classfp10,mtfcc10,csafp10,cbsafp10,funcstat10,aland10,awater10,intptlat10,intptlon10,year_conve,decade,decade_str,last_edite,last_edi_1,geometry,source_row
0,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07185.jpg\t,bxq07185,46.964901,-94.303902,1939.0,1,1,85,0.0,85,27,021,00659456,27021,Cass,Cass County,06,H1,G4020,NaN,14660,A,5.235770e+09,1.016740e+09,+46.9514271,-094.3337731,1939-01-01,1930,1930,BORCHERT,2026-07-31,POINT (-10497862.317 5936346.903),0
1,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07187.jpg\t,bxq07187,46.965599,-94.346802,1939.0,2,2,85,0.0,85,27,021,00659456,27021,Cass,Cass County,06,H1,G4020,NaN,14660,A,5.235770e+09,1.016740e+09,+46.9514271,-094.3337731,1939-01-01,1930,1930,BORCHERT,2026-07-31,POINT (-10502637.934 5936460.776),1
2,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07189.jpg\t,bxq07189,46.966202,-94.389801,1939.0,3,3,85,0.0,85,27,021,00659456,27021,Cass,Cass County,06,H1,G4020,NaN,14660,A,5.235770e+09,1.016740e+09,+46.9514271,-094.3337731,1939-01-01,1930,1930,BORCHERT,2026-07-31,POINT (-10507424.583 5936559.09),2
3,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07191.jpg\t,bxq07191,46.966801,-94.432800,1939.0,4,4,85,0.0,85,27,021,00659456,27021,Cass,Cass County,06,H1,G4020,NaN,14660,A,5.235770e+09,1.016740e+09,+46.9514271,-094.3337731,1939-01-01,1930,1930,BORCHERT,2026-07-31,POINT (-10512211.244 5936656.784),3
4,bxq,https://geo.lib.umn.edu/dnr_historical_aerial_imagery/airphotos/usda/bxq/y1939/bxq07195.jpg\t,bxq07195,46.967999,-94.518700,1939.0,5,5,85,0.0,85,27,021,00659456,27021,Cass,Cass County,06,H1,G4020,NaN,14660,A,5.235770e+09,1.016740e+09,+46.9514271,-094.3337731,1939-01-01,1930,1930,BORCHERT,2026-07-31,POINT (-10521773.51 5936852.176),4


In [23]:
# Remove leading and trailing whitespace.

text_fields = [
    "county",
    "flight_code",
    "photo_id",
    "url",
]

for col in text_fields:
    gdf[col] = gdf[col].astype("string").str.strip()

# Convert the year to a nullable integer.
gdf["year"] = pd.to_numeric(
    gdf["year"],
    errors="raise"
).astype("Int64")

In [26]:
# check if we're missing any of the things we need

required_fields = [
    "county",
    "year",
    "photo_id",
    "url",
    "flight_code",
]

missing_summary = pd.DataFrame({
    "missing": gdf[required_fields].isna().sum(),
    "percent": (
        gdf[required_fields].isna().mean() * 100
    ).round(2),
})

display(missing_summary)

,missing,percent
county,4680,4.04
year,0,0.00
photo_id,0,0.00
url,0,0.00
flight_code,0,0.00


### Populating missing county names
The county column is the only one missing values. Let's examine and populate this field. 

In [ ]:
# examine missing counties

missing_counties = gdf.loc[gdf["county"].isna()]

print("Records missing county value:", len(missing_counties))

Records missing county: 4680


In [ ]:
# identify county name column
display(counties.columns)
counties["co_name"].head()

Index(['objectid', 'co_name', 'st_fips', 'co_fips', 'co_code', 'co_gnis',
       'shape_leng', 'shape_Le_1', 'shape_Area', 'geometry'],
      dtype='str')

0      Aitkin
1       Anoka
2      Becker
3    Beltrami
4      Benton
Name: co_name, dtype: str

In [31]:
COUNTY_NAME_FIELD = "co_name"

# Create a smaller GeoDataFrame containing only
# the county name and polygon geometry.
county_lookup = counties[[COUNTY_NAME_FIELD, "geometry"]].copy()

# Rename the county field so we can distinguish it
# from the existing county field after the spatial join.
county_lookup = county_lookup.rename(
    columns={COUNTY_NAME_FIELD: "county_spatial"}
)

# Match the CRS of the photo points.
county_lookup = county_lookup.to_crs(gdf.crs)

In [ ]:
# spatially join photo points to county polygons
county_join = gpd.sjoin(
    gdf,
    county_lookup,
    how="left",
    predicate="within"
)

In [ ]:
# make sure no duplicates were created by spatial join
assert len(county_join) == len(gdf)

In [ ]:
# create dataframe with just county names from the 2 sources
# only where the county is NOT null 
county_comparison = county_join.loc[
    county_join["county"].notna()
    & county_join["county_spatial"].notna()
].copy()

In [ ]:
# check for mismatches between existing non-null and spatially joined county values
county_mismatches = county_comparison.loc[
    county_comparison["county"]
    != county_comparison["county_spatial"]
]

print("Records compared:", len(county_comparison))
print("County mismatches:", len(county_mismatches))

Records compared: 104517
County mismatches: 25145


In [ ]:
# examine this list
with pd.option_context("display.max_rows", None):
    display(
        county_mismatches
        .groupby(["county", "county_spatial"])
        .size()
        .sort_values(ascending=False)
        .reset_index(name="count")
    )

,county,county_spatial,count
0,St. Louis,Saint Louis,25039
1,Pope,Stearns,5
2,Steele,Dodge,5
3,Goodhue,Dakota,4
4,Carver,Hennepin,4
5,Anoka,Hennepin,4
6,Beltrami,Koochiching,4
7,Pope,Stevens,3
8,Beltrami,Clearwater,3
9,Morrison,Stearns,2


Besides St. Louis County, most of these mismatches apply to a very small number of points. Only 106 points are spatially outside of their labeled county. Based on these results, I think it's safe to populate the missing county values with the name of their spatially joined county. 